## Section 1: Setup and Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

matplotlib.use('Agg')
os.makedirs('outputs/eda_plots', exist_ok=True)

df = pd.read_parquet('outputs/final_dataset.parquet')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names (first 20):")
print(df.columns[:20].tolist())
print(f"\nColumn names (last 5):")
print(df.columns[-5:].tolist())
print(f"\nData types:")
print(df.dtypes.value_counts())

## Section 2: Missing Value Analysis

In [ ]:
print(f"Total rows: {len(df):,}")
print()

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    missing_df = pd.DataFrame({
        'column': missing.index,
        'missing_count': missing.values,
        'missing_pct': (missing.values / len(df) * 100).round(2)
    })
    print("Columns with missing values:")
    print(missing_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.4)))
    ax.barh(missing_df['column'], missing_df['missing_pct'], color='steelblue')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/eda_plots/missing_values.png', dpi=150)
    plt.close()
    print("\nSaved: outputs/eda_plots/missing_values.png")

## Section 3: Target Variable Distribution

In [ ]:
vc = df['readmitted_30d'].value_counts().sort_index()
rate = df['readmitted_30d'].mean() * 100

print("readmitted_30d value counts:")
print(vc)
print(f"\nReadmission rate: {rate:.2f}%")

labels = ['Not Readmitted (0)', 'Readmitted (1)']
counts = [vc.get(0, 0), vc.get(1, 0)]
pcts = [c / len(df) * 100 for c in counts]
colors = ['#4c72b0', '#dd8452']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts, color=colors, width=0.5)
for bar, cnt, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{cnt:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Count')
ax.set_title('30-Day Readmission Distribution')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig('outputs/eda_plots/target_distribution.png', dpi=150)
plt.close()
print("Saved: outputs/eda_plots/target_distribution.png")

## Section 4: Clinical Feature Distributions

In [ ]:
numeric_cols = ['age_at_admission', 'los_days', 'prior_admissions_count']
numeric_cols = [c for c in numeric_cols if c in df.columns]

for col in numeric_cols:
    s = df[col].dropna().astype(float)
    mean, std = s.mean(), s.std()
    outliers = (s > mean + 3 * std).sum()

    print(f"--- {col} ---")
    print(f"  min   : {s.min():.2f}")
    print(f"  max   : {s.max():.2f}")
    print(f"  mean  : {mean:.2f}")
    print(f"  median: {s.median():.2f}")
    print(f"  std   : {std:.2f}")
    print(f"  values > 3 std above mean: {outliers:,}")
    print()

    fig, ax = plt.subplots(figsize=(8, 4))
    for label, grp in df.groupby('readmitted_30d'):
        vals = grp[col].dropna().astype(float)
        vals.plot.hist(ax=ax, alpha=0.5, bins=40, density=True,
                       label=f"readmitted={int(label)}")
        vals.plot.kde(ax=ax, label='')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {col} by Readmission')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_dist.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_dist.png")

## Section 5: Categorical Feature Analysis

In [ ]:
cat_cols = ['gender', 'admission_type', 'insurance', 'race',
            'discharge_location', 'marital_status']
cat_cols = [c for c in cat_cols if c in df.columns]

for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    pct = (vc / len(df) * 100).round(2)
    summary = pd.DataFrame({'count': vc, 'pct': pct})

    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(summary.to_string())
    print()

    plot_vc = vc.sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, len(plot_vc) * 0.35)))
    ax.barh(plot_vc.index.astype(str), plot_vc.values, color='steelblue')
    ax.set_xlabel('Count')
    ax.set_title(f'{col} - Value Counts')
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_counts.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_counts.png\n")

## Section 6: Readmission Rate by Category

In [ ]:
for col in cat_cols:
    rate_df = (
        df.groupby(col, dropna=False)['readmitted_30d']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'readmission_rate', 'count': 'n'})
        .sort_values('readmission_rate', ascending=False)
    )
    rate_df['readmission_rate_pct'] = (rate_df['readmission_rate'] * 100).round(2)

    print(f"--- {col} ---")
    print(rate_df[['n', 'readmission_rate_pct']].to_string())
    print()

    plot_df = rate_df.sort_values('readmission_rate', ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, len(plot_df) * 0.35)))
    ax.barh(plot_df.index.astype(str), plot_df['readmission_rate_pct'], color='#dd8452')
    ax.set_xlabel('Readmission Rate (%)')
    ax.set_title(f'Readmission Rate by {col}')
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_readmission_rate.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_readmission_rate.png\n")

## Section 7: CCSR Feature Summary

In [ ]:
non_ccsr = {
    'subject_id', 'hadm_id', 'gender', 'age_at_admission',
    'admission_type', 'insurance', 'race', 'discharge_location',
    'marital_status', 'los_days', 'came_via_ed',
    'prior_admissions_count', 'days_to_next', 'readmitted_30d'
}
ccsr_cols = [c for c in df.columns if c not in non_ccsr]

print(f"Total CCSR columns: {len(ccsr_cols)}")

empty_ccsr = [c for c in ccsr_cols if df[c].sum() == 0]
print(f"CCSR columns with all zeros (no patient has this condition): {len(empty_ccsr)}")

prevalence = df[ccsr_cols].mean().sort_values(ascending=False)

print("\nTop 20 most prevalent CCSR categories:")
top20 = prevalence.head(20)
for cat, prev in top20.items():
    print(f"  {cat:<20}  {prev*100:.2f}%")

nonzero_prev = prevalence[prevalence > 0]
bottom10 = nonzero_prev.tail(10)
print("\nBottom 10 least prevalent (nonzero) CCSR categories:")
for cat, prev in bottom10.items():
    print(f"  {cat:<20}  {prev*100:.4f}%")

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(top20)), top20.values * 100, color='steelblue')
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index.tolist(), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Top 20 Most Prevalent CCSR Diagnosis Categories')
plt.tight_layout()
plt.savefig('outputs/eda_plots/ccsr_top20_prevalence.png', dpi=150)
plt.close()
print("\nSaved: outputs/eda_plots/ccsr_top20_prevalence.png")

## Section 8: Class Imbalance and Correlation Check

In [ ]:
from scipy.stats import pointbiserialr

target_vc = df['readmitted_30d'].value_counts()
majority = target_vc.max()
minority = target_vc.min()
print(f"Class balance ratio (majority / minority): {majority / minority:.2f}:1")
print(f"  Class 0: {target_vc.get(0, 0):,}")
print(f"  Class 1: {target_vc.get(1, 0):,}")
print()

num_cols = ['age_at_admission', 'los_days', 'prior_admissions_count']
num_cols = [c for c in num_cols if c in df.columns]

corr_results = []
for col in num_cols:
    valid = df[[col, 'readmitted_30d']].dropna()
    r, p = pointbiserialr(valid[col].astype(float), valid['readmitted_30d'].astype(float))
    corr_results.append({'feature': col, 'correlation': round(r, 4), 'p_value': round(p, 6)})

corr_df = pd.DataFrame(corr_results).sort_values('correlation', key=abs, ascending=False)
print("Point-biserial correlation with readmitted_30d:")
print(corr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#4c72b0' if v >= 0 else '#dd8452' for v in corr_df['correlation']]
ax.barh(corr_df['feature'], corr_df['correlation'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point-Biserial Correlation')
ax.set_title('Numeric Feature Correlation with Readmission')
plt.tight_layout()
plt.savefig('outputs/eda_plots/numeric_correlations.png', dpi=150)
plt.close()
print("\nSaved: outputs/eda_plots/numeric_correlations.png")

## Section 9: EDA Summary Report

In [ ]:
non_ccsr_set = {
    'subject_id', 'hadm_id', 'gender', 'age_at_admission',
    'admission_type', 'insurance', 'race', 'discharge_location',
    'marital_status', 'los_days', 'came_via_ed',
    'prior_admissions_count', 'days_to_next', 'readmitted_30d'
}
_ccsr_cols = [c for c in df.columns if c not in non_ccsr_set]
_clinical_cols = [c for c in df.columns if c not in non_ccsr_set and c not in ('subject_id', 'hadm_id', 'readmitted_30d')]

_missing = df.isnull().sum()
_missing = _missing[_missing > 0]

_vc = df['readmitted_30d'].value_counts().sort_index()
_n0 = int(_vc.get(0, 0))
_n1 = int(_vc.get(1, 0))
_rate = df['readmitted_30d'].mean() * 100

_empty_ccsr = [c for c in _ccsr_cols if df[c].sum() == 0]
_prev = df[_ccsr_cols].mean().sort_values(ascending=False)
_top3 = _prev.head(3)

_ccsr_quoted = [c for c in _ccsr_cols if c.startswith("'") or c.endswith("'")]
_died_rows = int((df['discharge_location'] == 'DIED').sum()) if 'discharge_location' in df.columns else 0
_race_unique = df['race'].nunique() if 'race' in df.columns else 'N/A'

print("=" * 60)
print("DATASET SUMMARY")
print(f"  Total rows              : {len(df):,}")
print(f"  Total columns           : {df.shape[1]}")
print(f"  Clinical feature columns: {len(_clinical_cols)}")
print(f"  CCSR diagnosis columns  : {len(_ccsr_cols)}")
print(f"  Target                  : readmitted_30d")
print()
print("MISSING VALUES")
if len(_missing) == 0:
    print("  None found.")
else:
    for col, cnt in _missing.items():
        print(f"  {col}: {cnt:,} missing ({cnt/len(df)*100:.2f}%)")
print()
print("TARGET")
print(f"  Not readmitted (0): {_n0:,} rows, {_n0/len(df)*100:.1f}%")
print(f"  Readmitted     (1): {_n1:,} rows, {_n1/len(df)*100:.1f}%")
print(f"  Readmission rate  : {_rate:.2f}%")
print()
print("CCSR FEATURES")
print(f"  Total CCSR columns         : {len(_ccsr_cols)}")
print(f"  Empty CCSR columns (all 0) : {len(_empty_ccsr)}")
print(f"  Top 3 most prevalent conditions:")
for cat, prev in _top3.items():
    print(f"    {cat}  ({prev*100:.2f}%)")
print()
print("ISSUES TO FIX IN PREPROCESSING")

issues = []
for col, cnt in _missing.items():
    issues.append(f"  - Missing values in '{col}': {cnt:,} rows -> impute or drop")

if len(_ccsr_quoted) > 0:
    issues.append(f"  - CCSR column names have embedded single quotes ({len(_ccsr_quoted)} cols) -> strip quotes and spaces in preprocessing")

if _died_rows > 0:
    issues.append(f"  - discharge_location = 'DIED': {_died_rows:,} rows -> consider excluding (patient deceased, readmission not applicable)")

issues.append(f"  - race has {_race_unique} unique values -> consolidate rare categories before encoding")

for issue in issues:
    print(issue)

print("=" * 60)